# 🏥 HealthConnect AI — Pipeline i Plotë i Machine Learning

Ky Jupyter Notebook përmban të gjithë procesin e zhvillimit për modelin e Inteligjencës Artificiale të integruar në **HealthConnect AI**:
1. **Leximi dhe Inspektimi** i Dataset-it (Pima Indians Diabetes)
2. **Preprocessing** (Zëvendësimi i vlerave 0 joreale me medianën dhe capping i outliers me IQR)
3. **Ndarja Train/Test (80/20)** dhe **Skalimi** (StandardScaler)
4. **Trajnimi i 5 Modeleve Supervised** (k-NN, Random Forest, Logistic Regression, 2x MLP)
5. **Cross-Validation** (Stratified 5-Fold) për të vlerësuar qëndrueshmërinë
6. **Grupimi Unsupervised** me K-Means (K=3) për zbulimin e profileve të rrezikut metabolik
7. **Testimi i Parashikimit** mbi një pacient shembull

## 🛠️ Konfigurimi i Mjedisit dhe Importet

In [1]:
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# Scikit-Learn imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, silhouette_score, adjusted_rand_score
)

warnings.filterwarnings('ignore')

# Konfigurimi i shtigjeve
BASE_DIR = Path(os.getcwd()).parent if 'ml' in os.getcwd() else Path(os.getcwd())
DATASETS_DIR = BASE_DIR / "datasets"
PROCESSED_DIR = DATASETS_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

print(f"Base Directory: {BASE_DIR}")
print(f"Datasets Directory: {DATASETS_DIR}")

KeyboardInterrupt: 

## 🔍 1. Leximi dhe Inspektimi i Dataset-it

In [ ]:
csv_path = DATASETS_DIR / "diabetes.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"Nuk u gjet dataset-i ne: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Dimensionet e dataset-it: {df.shape[0]} rreshta x {df.shape[1]} kolona")
print("\n5 Rreshtat e parë:")
print(df.head())

print("\nStatistikat Përmbledhëse:")
print(df.describe().round(2))

### 1.1 Balanca e Klasave dhe Analiza e Vlerave Munguese (Zeros)

In [ ]:
print("Balanca e klasës target 'Outcome':")
counts = df['Outcome'].value_counts()
for cls, count in counts.items():
    pct = count / len(df) * 100
    print(f"  Klasa {cls} (0=Jo Diabet, 1=Diabet): {count} pacientë ({pct:.1f}%)")

print("\nAnaliza e vlerave zero në kolona joreale:")
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_with_zeros:
    n_zeros = (df[col] == 0).sum()
    print(f"  - Kolona {col:<20} ka {n_zeros} vlera zero (0)")

## 🧼 2. Preprocessing: Zëvendësimi i Zero me Medianë dhe IQR Capping

In [ ]:
df_clean = df.copy()

# 2.1 Median Imputation bazuar në klasën target
print("Zëvendësimi i zero me medianën e klasës përkatëse:")
for col in cols_with_zeros:
    df_clean[col] = df_clean[col].astype(float)
    for outcome in [0, 1]:
        mask = (df_clean[col] == 0) & (df_clean['Outcome'] == outcome)
        median_val = df_clean.loc[(df_clean[col] != 0) & (df_clean['Outcome'] == outcome), col].median()
        df_clean.loc[mask, col] = median_val
    print(f"  [OK] Zëvendësuar zeros në: {col}")

# 2.2 IQR Outliers Capping
print("\nCapping i vlerave ekstreme (Outliers) me IQR:")
for col in df_clean.columns:
    if col == 'Outcome' or df_clean[col].nunique() <= 5:
        continue
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    n_capped = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    df_clean[col] = df_clean[col].clip(lower, upper)
    if n_capped > 0:
        print(f"  - {col:<25}: Capped {n_capped} outliers")

## 📊 3. Ndarja Train/Test dhe Skalimi i të Dhënave

In [ ]:
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f"Train Set: {X_train.shape[0]} pacientë")
print(f"Test Set : {X_test.shape[0]} pacientë")

# StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# Ruajmë skedarët
train_save = X_train_scaled.copy()
train_save['Outcome'] = y_train.values
test_save = X_test_scaled.copy()
test_save['Outcome'] = y_test.values

train_save.to_csv(PROCESSED_DIR / "diabetes_train.csv", index=False)
test_save.to_csv(PROCESSED_DIR / "diabetes_test.csv", index=False)

# Ruajmë skalerin me pickle
with open(PROCESSED_DIR / "scalers.pkl", 'wb') as f:
    pickle.dump({'diabetes': scaler}, f)
print("\n[OK] Të dhënat u ndanë, u skaluan dhe u ruajtën me sukses!")

## 🤖 4. Trajnimi dhe Vlerësimi i 5 Modeleve Supervised

In [ ]:
models = {
    "kNN (k=5)": KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "MLP Neural Network 1": MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=RANDOM_STATE, early_stopping=True),
    "MLP Neural Network 2": MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=500, random_state=RANDOM_STATE, early_stopping=True, learning_rate='adaptive')
}

results = []
best_f1 = 0
best_model = None
best_model_name = ""

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    print(f"Modeli: {name:<22} | Accuracy: {acc*100:.2f}% | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f}")
    
    # Ruajmë skedarin .pkl për çdo model
    safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("=", "_")
    with open(MODELS_DIR / f"diabetes_{safe_name}.pkl", "wb") as f:
        pickle.dump(model, f)
        
    results.append({"Model": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1})
    
    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_model_name = name

print(f"\n🏆 Modeli më i mirë: {best_model_name} me F1-Score: {best_f1:.4f}")
# Ruajmë modelin më të mirë si modelin kryesor për Backend
with open(MODELS_DIR / "diabetes_production_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

## 🔄 5. Stratified 5-Fold Cross-Validation

In [ ]:
print("Cross-Validation me Stratified 5-Fold (F1-score):")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='f1', n_jobs=-1)
    print(f"  - {name:<22} | F1-Mean: {scores.mean():.4f} ± {scores.std():.4f}")

## 🎯 6. Grupimi Unsupervised me K-Means Clustering (K=3)

In [ ]:
K = 3
km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20)

# Bashkojmë të dhënat train + test të skaluara për clustering të plotë
X_full_scaled = pd.concat([X_train_scaled, X_test_scaled], ignore_index=True)
y_full = pd.concat([y_train, y_test], ignore_index=True)

km.fit(X_full_scaled)
sil = silhouette_score(X_full_scaled, km.labels_)
ars = adjusted_rand_score(y_full, km.labels_)

print(f"Silhouette Score    : {sil:.4f}")
print(f"Adjusted Rand Index : {ars:.4f}")

# Renditim clusterët sipas Glukozës mesatare për të patur nivele të kuptueshme rreziku
df_clust = X_full_scaled.copy()
df_clust['cluster'] = km.labels_
df_clust['Outcome'] = y_full.values

glucose_means = df_clust.groupby('cluster')['Glucose'].mean().sort_values()
cluster_order = glucose_means.index.tolist()

risk_labels = {
    cluster_order[0]: 'Rrezik i Ulet',
    cluster_order[1]: 'Rrezik Mesatar',
    cluster_order[2]: 'Rrezik i Larte'
}

print("\nKarakteristikat e Cluster-ave të zbuluar:")
for cid in cluster_order:
    subset = df_clust[df_clust['cluster'] == cid]
    diab_pct = subset['Outcome'].mean() * 100
    print(f"  * Cluster {cid} ({risk_labels[cid]}): {len(subset)} pacientë | {diab_pct:.1f}% janë diabetikë")

# Ruajmë modelin e K-Means
with open(MODELS_DIR / "diabetes_kmeans.pkl", "wb") as f:
    pickle.dump({
        'model': km,
        'risk_labels': risk_labels,
        'feature_names': X.columns.tolist()
    }, f)

## 🧪 7. Testimi i Parashikimit mbi një Pacient të Ri

In [ ]:
# Të dhënat e një pacienti të ri
new_patient = {
    "Pregnancies": 6,
    "Glucose": 148.0,
    "BloodPressure": 72.0,
    "SkinThickness": 35.0,
    "Insulin": 120.0,
    "BMI": 33.6,
    "DiabetesPedigreeFunction": 0.627,
    "Age": 50
}

feature_names = X.columns.tolist()
df_new = pd.DataFrame([new_patient])[feature_names]

# Skalimi
df_new_scaled = pd.DataFrame(scaler.transform(df_new), columns=feature_names)

# Parashikimi me modelin tonë më të mirë (Random Forest)
rf_model = models["Random Forest"]
pred = int(rf_model.predict(df_new_scaled)[0])
prob = float(rf_model.predict_proba(df_new_scaled)[0][1])

# Grupimi me K-Means
cid = km.predict(df_new_scaled)[0]
risk = risk_labels[cid]

print("Rezultati i Parashikimit:")
print(f"  - Parashikimi i Diabetit : {pred} (1 = Po, 0 = Jo)")
print(f"  - Probabiliteti          : {prob*100:.1f}%")
print(f"  - Grupi i Rrezikut       : {risk}")